# NIST TN 1822 — Verif.1.1: Pre-evacuation time distributions

Verify the model applies each pre-evacuation distribution correctly. Section 3.1.1; printed page 16 / PDF page 22.

Reference: <https://nvlpubs.nist.gov/nistpubs/technicalnotes/NIST.TN.1822.pdf>

In [ ]:
from datetime import datetime
print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from shapely.geometry import Point, Polygon

from jupedsim_scenarios import load_scenario, run_scenario

In [ ]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f5",
    "axes.edgecolor": "#3a3a3a",
    "axes.labelcolor": "#1d1d1d",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.figsize": (8, 5),
})

In [ ]:
from scenario_builders.nist1_1_premovement import build_variants, sample_reference

## Load the base scenario

In [ ]:
SCENARIO_ZIP = Path("scenario_files") / "Nist-1-1-premovement.zip"
base = load_scenario(str(SCENARIO_ZIP))
print(base.summary())

## Sweep the four NIST distributions

`build_variants` deep-copies the base scenario, overrides `premovement_*` via `set_agent_params`, and bumps `max_time` to a value appropriate for the distribution's tail.

In [ ]:
runs = {}
for case, variant in build_variants(base):
    result = run_scenario(variant, seed=42)
    df = result.trajectory_dataframe()
    # First non-stationary frame per agent = observed start time.
    starts = []
    for agent_id, sub in df.sort_values(['id', 'frame']).groupby('id'):
        x0, y0 = sub.iloc[0][['x', 'y']]
        moved = sub[(sub.x - x0).abs() + (sub.y - y0).abs() > 0.05]
        t = (moved.iloc[0]['frame'] / result.frame_rate) if len(moved) else float('nan')
        starts.append(t)
    runs[case.name] = {'case': case, 'observed_starts': np.array(starts)}
    result.cleanup()

## Plot observed vs analytic

`sample_reference` re-uses the upstream distribution sampler so the overlay is by definition the same family the loader applied.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (name, run) in zip(axes.flat, runs.items()):
    obs = run['observed_starts']
    obs = obs[~np.isnan(obs)]
    if len(obs):
        ax.hist(obs, bins=20, density=True, alpha=0.6, label='observed')
    ref = sample_reference(run['case'], 10000, seed=1)
    ax.hist(ref, bins=80, density=True, histtype='step', label='reference')
    ax.set_title(name)
    ax.set_xlabel('pre-evac time [s]')
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Acceptance

In [ ]:
from scipy import stats as _stats
ALPHA = 0.05
rows = []
for name, run in runs.items():
    obs = run['observed_starts']
    obs = obs[~np.isnan(obs)]
    ref = sample_reference(run['case'], 10000, seed=2)
    ks = _stats.ks_2samp(obs, ref) if len(obs) else None
    rows.append({
        'case': name, 'n': int(len(obs)),
        'ks_p': ks.pvalue if ks else float('nan'),
    })
fit = pd.DataFrame(rows)
print(fit)
assert (fit['ks_p'] > ALPHA).all(), fit